# Урок 16. Обработка данных из файла

9 класс · II четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 15](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-15.ipynb) · [Урок 17 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-17.ipynb)

---

Разбор строк файла в числа и поля. Накопление статистики. Отбор строк по условию. Вывод отчёта.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 9А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-16", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Перепись, которую считали машиной

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/g09/holerit.jpg" width="300" alt="Табулятор Холлерита, 1902 год">

*Табулятор Холлерита, 1902 год*

<sub>Бюро переписи США · общественное достояние · Wikimedia Commons</sub>

Перепись населения США 1880 года обрабатывали вручную восемь лет —
результаты устарели раньше, чем их напечатали. К переписи 1890 года
Герман Холлерит предложил другое: каждого человека закодировать
дырками на картонной карте, а карты пропустить через машину со щётками
и счётчиками.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/g09/holerit-karta.jpg" width="320" alt="Перфокарта Холлерита">

*Перфокарта Холлерита*

<sub>автор неизвестен · общественное достояние · Wikimedia Commons</sub>

Каждая позиция на карте означала признак: возраст, пол, профессию.
Машина считала не людей, а сочетания дырок — и справилась за год.
Компания Холлерита позже стала называться IBM.

Сегодня перфокарту заменил файл, а щётки — цикл `for`. Идея осталась
прежней: **одна строка файла — одна запись, поля внутри строки
разделены известным символом**.

### Строка данных

```
Аня;информатика;5
Боря;математика;4
Вика;информатика;3
```

* **запись** — одна строка;
* **поле** — часть записи: имя, предмет, оценка;
* **разделитель** — символ между полями: точка с запятой, запятая
  или табуляция.

Файл, где поля разделены запятыми, называют **csv** — от английского
comma-separated values. Его открывают и электронные таблицы, и любая
программа: это самый распространённый способ передать таблицу данных.

### Схема обработки

Почти любая задача на файл с данными решается по одному шаблону:

```
1. подготовить накопители (сумма = 0, счётчик = 0, словарь = {})
2. для каждой строки файла:
       убрать перевод строки        строка.strip()
       разрезать на поля            строка.split(";")
       превратить числа в числа     int(...) или float(...)
       отобрать нужные записи       if ...
       обновить накопители
3. посчитать итоги и вывести отчёт
```

Разбор строки удобно писать распаковкой: `имя, предмет, оценка =
строка.split(";")` — если полей ровно три, получаются сразу три
переменные с понятными именами.

### Частая ошибка

Всё, что прочитано из файла, — **строки**. `"5" + "4"` даст `"54"`,
а не 9. Числа нужно превращать в числа сразу при разборе строки,
и тогда дальше об этом можно не думать.

## Смотрим, как это работает

### Пример 1. Готовим файл с данными

Обычно такой файл берут готовым, но нам нужно с чего-то начать.

In [ ]:
данные = """Аня;информатика;5
Боря;математика;4
Вика;информатика;3
Аня;математика;5
Гриша;информатика;4
Боря;информатика;5
Вика;математика;4
Гриша;математика;2
"""

with open("журнал.txt", "w", encoding="utf-8") as файл:
    файл.write(данные)

with open("журнал.txt", "r", encoding="utf-8") as файл:
    print(файл.read())

Тройные кавычки позволяют записать в строку сразу несколько строк —
переносы внутри сохраняются.

### Пример 2. Разбор и статистика

In [ ]:
записи = []
with open("журнал.txt", "r", encoding="utf-8") as файл:
    for строка in файл:
        строка = строка.strip()
        if not строка:
            continue
        имя, предмет, оценка = строка.split(";")
        записи.append((имя, предмет, int(оценка)))

print("Записей:", len(записи))
print("Первая:", записи[0])

оценки = [оценка for имя, предмет, оценка in записи]
print("Средний балл по всем:", round(sum(оценки) / len(оценки), 2))
print("Пятёрок:", оценки.count(5))

Сначала весь файл превращён в список записей, и только потом считается
статистика. Это удобно, когда данных немного: разобрали один раз,
а считать можно что угодно.

### Пример 3. Отбор по условию

In [ ]:
информатика = [запись for запись in записи if запись[1] == "информатика"]

print("Оценки по информатике:")
for имя, предмет, оценка in информатика:
    print(f"  {имя:<7} {оценка}")

баллы = [оценка for имя, предмет, оценка in информатика]
print("Средний балл по информатике:", round(sum(баллы) / len(баллы), 2))

### Пример 4. Сводка по каждому ученику

Здесь без словаря уже не обойтись: заранее неизвестно, сколько будет
учеников и как их зовут.

In [ ]:
по_ученикам = {}
for имя, предмет, оценка in записи:
    if имя not in по_ученикам:
        по_ученикам[имя] = []
    по_ученикам[имя].append(оценка)

строки_отчёта = []
for имя in sorted(по_ученикам):
    оценки_ученика = по_ученикам[имя]
    средний = sum(оценки_ученика) / len(оценки_ученика)
    строки_отчёта.append(f"{имя:<7} оценок: {len(оценки_ученика)}  средний: {средний:.2f}")

отчёт = "\n".join(строки_отчёта)
print(отчёт)

with open("сводка.txt", "w", encoding="utf-8") as файл:
    файл.write(отчёт + "\n")

Три строки в середине — стандартный приём «накопить список по ключу»:
если ключа ещё нет, заводим пустой список, и только потом добавляем.

### Пример 5. Лучший ученик

In [ ]:
средние = {}
for имя, оценки_ученика in по_ученикам.items():
    средние[имя] = sum(оценки_ученика) / len(оценки_ученика)

лучший = max(средние, key=lambda имя: средние[имя])
print("Лучший средний балл у:", лучший, round(средние[лучший], 2))

`max` по словарю перебирает ключи, а `key` говорит, по какому числу
их сравнивать. Если максимум делят двое, вернётся тот, кто встретился
раньше.

## Пробуем сами

Все задачи — по файлу `журнал.txt` из примера 1.

### Задача 1. Как называется строка файла с данными

In [ ]:
#@title 🧩 Задача 1. Термин { display-mode: "form" }
#@markdown Одна строка таблицы данных — это…
термин = "выбери ответ" #@param ["выбери ответ", "поле", "запись", "разделитель"]

si.ответ("1", термин, "1fa00777a7ee735a",
         hint="Поле — это часть строки, а строка целиком?")

### Задача 2. Сколько записей

Функция возвращает количество непустых записей в файле.

In [ ]:
def записей(имя_файла):
    return ...

In [ ]:
si.check("2", записей, [
    ("журнал.txt", 8),
])

### Задача 3. Средний балл по предмету

Функция получает имя файла и название предмета, возвращает средний
балл по этому предмету, округлённый до двух знаков. Если записей
по предмету нет — 0.

In [ ]:
def средний_по_предмету(имя_файла, предмет_нужный):
    return ...

In [ ]:
si.check("3", средний_по_предмету, [
    (("журнал.txt", "информатика"), 4.25),
    (("журнал.txt", "математика"), 3.75),
    (("журнал.txt", "физика"), 0),
])

### Задача 4. Оценки одного ученика

Функция возвращает список оценок ученика в том порядке, в котором они
записаны в файле.

In [ ]:
def оценки_ученика(имя_файла, кто):
    return ...

In [ ]:
si.check("4", оценки_ученика, [
    (("журнал.txt", "Аня"), [5, 5]),
    (("журнал.txt", "Гриша"), [4, 2]),
    (("журнал.txt", "Никто"), []),
])

### Задача 5. Сколько двоек и троек

Функция возвращает количество оценок ниже четырёх во всём файле.

In [ ]:
def слабых_оценок(имя_файла):
    return ...

In [ ]:
si.check("5", слабых_оценок, [
    ("журнал.txt", 2),
])

### Задача 6. Сводка по ученикам

Функция возвращает словарь «имя → список оценок».

In [ ]:
def сводка(имя_файла):
    return ...

In [ ]:
si.check("6", сводка, [
    ("журнал.txt", {"Аня": [5, 5], "Боря": [4, 5], "Вика": [3, 4], "Гриша": [4, 2]}),
])

### Задача 7. Кто лучший

По файлу `журнал.txt`: у кого самый высокий средний балл?
Посчитайте программой из примера 5 или вручную.

In [ ]:
#@title 🧩 Задача 7. Лучший ученик { display-mode: "form" }
#@markdown Выберите ответ
лучший_ученик = "выбери ответ" #@param ["выбери ответ", "Аня", "Боря", "Вика", "Гриша"]

si.ответ("7", лучший_ученик, "306297f0f739bfb4",
         hint="У кого обе оценки пятёрки?")

## Домашнее задание

### Домашнее задание 1. Отличники

Функция возвращает список имён тех, у кого все оценки — пятёрки.
Имена в алфавитном порядке, повторов быть не должно.

In [ ]:
def отличники(имя_файла):
    return ...

In [ ]:
si.check("дз1", отличники, [
    ("журнал.txt", ["Аня"]),
])

### Домашнее задание 2. Отчёт в файл

Функция считает средний балл по каждому предмету, записывает отчёт
в новый файл (по строке на предмет, предметы в алфавитном порядке,
формат `предмет: средний`) и возвращает количество строк отчёта.

Пример строки: `информатика: 4.25`

In [ ]:
def отчёт_по_предметам(имя_файла, куда):
    return ...

In [ ]:
si.check("дз2", отчёт_по_предметам, [
    (("журнал.txt", "по_предметам.txt"), 2),
])

Откройте получившийся файл и проверьте, что в нём написано:

```python
with open("по_предметам.txt", encoding="utf-8") as файл:
    print(файл.read())
```

### Домашнее задание 3. Свои данные

Соберите свой файл данных: минимум пятнадцать строк из трёх полей.
Что угодно — расходы за неделю, результаты матчей, время сна.
Напишите программу, которая отвечает на три вопроса по этим данным,
и запишите ответы в файл отчёта. На уроке покажете и объясните,
что считали.

---

### Любопытно

Перфокарты пережили Холлерита почти на век: ещё в 1980-х студенты
сдавали программы стопками карт, а одна выпавшая карта означала
сломанную программу. Отсюда привычка нумеровать строки — номер писали
в последних колонках карты, чтобы рассыпанную стопку можно было
собрать обратно машиной-сортировщиком.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 15](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-15.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 17 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-17.ipynb)